#파이썬으로 저장, 조회

SQLEditor에서 테스트한 SQL문
- insert, update, delete, select문을 코드로 문장을 코드로 실행한다

In [ ]:
from db import supabase

In [4]:
import os

from dotenv import load_dotenv
from supabase import Client, create_client

load_dotenv()

SUPABASE_URL = os.environ["SUPABASE_URL"]
SUPABASE_SERVICE_ROLE_KEY = os.environ["SUPABASE_SERVICE_ROLE_KEY"]

supabase: Client = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)

## 수퍼베이 설치확임

In [6]:
from supabase import Client, create_client

In [7]:
sup_client = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)

In [11]:
users_reault = sup_client.table("users").select("*",count="exact").execute()
users_reault.count


5

## 수퍼베이스 연결 모듈(db.py) 호출

In [12]:
from db import supabase

In [14]:
users_reault = supabase.table("users").select("*",count="exact").execute()
users_reault.count

5

In [15]:
#inset into users

supabase.table("users").insert(
    {
        "email":"asdasd@naver.com",
        "username":"윤동희"
    }
).execute()

APIResponse(data=[{'id': 7, 'email': 'asdasd@naver.com', 'username': '윤동희', 'created_at': '2026-08-24T03:19:22.044246'}], count=None)

In [ ]:
## where 조건 조회 : .eq

result2 = supabase.table("users").select("*").eq("username", "윤동희").execute()

print(result2)

data=[{'id': 7, 'email': 'asdasd@naver.com', 'username': '윤동희', 'created_at': '2026-08-24T03:19:22.044246'}] count=None


In [21]:
# 사용자 추가
result = supabase.table("users").insert({
    "email": "sql_to_py@example.com",
    "username": "파이썬",
}).execute()

In [22]:
new_user = result.data[0]
new_user_id = new_user["id"]     # DB 가 만들어준 id. 아래에서 쓴다.

print("추가된 사용자:", new_user)

추가된 사용자: {'id': 8, 'email': 'sql_to_py@example.com', 'username': '파이썬', 'created_at': '2026-08-24T03:32:58.485149'}


In [24]:
## 대화와 메세지 추가

#대화 1건
result = supabase.table("conversations").insert({
    "user_id": new_user_id,
    "title": "파이썬으로 만든 대화",
}).execute()

print(result)

data=[{'id': 9, 'user_id': 8, 'title': '파이썬으로 만든 대화', 'created_at': '2026-08-24T03:37:26.785782+00:00'}] count=None


In [25]:
conversation_id = result.data[0]["id"]
print("대화 생성:", result.data[0]["title"])

# 메시지 2건을 한 번에. 여러 건은 리스트로 넘긴다.
result = supabase.table("messages").insert([
    {"conversation_id": conversation_id, "role": "user", "content": "파이썬에서도 저장이 되나요?"},
    {"conversation_id": conversation_id, "role": "assistant", "content": "네, 방금 저장됐습니다."},
]).execute()

print("메시지", len(result.data), "건 저장")

대화 생성: 파이썬으로 만든 대화
메시지 2 건 저장


In [26]:
# LEFT JOIN 에 해당
result = supabase.table("users").select("username, conversations(title)").execute()

for user in result.data:
    print(" ", user["username"])
    if len(user["conversations"]) == 0:
        print("     (대화 없음)")
    for conversation in user["conversations"]:
        print("    -", conversation["title"])


  테스터
    - 파이썬 기초 질문
  권지용
    - 이직 고민 상담
  강대성
    - SQL 공부 방법
  동태양
    - 여행 계획 짜기
  라리사
     (대화 없음)
  윤동희
     (대화 없음)
  파이썬
    - 파이썬으로 만든 대화
    - 파이썬으로 만든 대화


In [27]:

# INNER JOIN 에 해당
result = supabase.table("users").select("username, conversations!inner(title)").execute()
print("!inner 를 쓰면:")
for user in result.data:
    print("   ", user["username"])

!inner 를 쓰면:
    테스터
    권지용
    강대성
    동태양
    파이썬


In [28]:
# 3단계 중첩
result = (
    supabase.table("users")
    .select("username, conversations(title, messages(role, content))")
    .eq("email", "sql_to_py@example.com")
    .execute()
)

for user in result.data:
    for conversation in user["conversations"]:
        print(" ", user["username"], "-", conversation["title"])
        for message in conversation["messages"]:
            print("    ", message["role"], ":", message["content"])

  파이썬 - 파이썬으로 만든 대화
  파이썬 - 파이썬으로 만든 대화
     user : 파이썬에서도 저장이 되나요?
     assistant : 네, 방금 저장됐습니다.
